In [1]:
import asyncio
import edge_tts
import os
import random
from pydub import AudioSegment
import nest_asyncio  # 解决嵌套循环问题

In [9]:
nest_asyncio.apply()

# --- 配置参数 ---
NOISE_FILE = "on_the_bus.wav" # 你的背景噪音文件
OUTPUT_ROOT = "test_experiment"
TRUE_DIR = os.path.join(OUTPUT_ROOT, "true")
FALSE_DIR = os.path.join(OUTPUT_ROOT, "false")
TARGET_DURATION_MS = 10000  # 10秒 = 10000毫秒
TOTAL_SAMPLES_PER_CLASS = 50

# --- 50 条包含 "Bob" 的语音 (True) ---
TRUE_TEMPLATES = [
    "Hello, is Bob there?", "Hey Bob, can you hear me?", "I'm calling to speak with Bob.", 
    "Bob, please pick up the phone.", "Is that you, Bob?", "Good morning, Bob.", 
    "Tell Bob I'm waiting.", "Can I talk to Bob for a second?", "Bob is my best friend.",
    "Where did Bob go?", "I saw Bob at the park yesterday.", "Please give this to Bob.",
    "Bob, we need to leave now.", "Is Bob available for a meeting?", "Ask Bob about the report.",
    "Bob, did you see the email?", "That sounds like something Bob would say.", 
    "Tell Bob the package arrived.", "Bob, can you help me with this?", "I haven't seen Bob lately.",
    "Call Bob as soon as possible.", "Bob, are you busy right now?", "It's time for Bob to join us.",
    "I believe Bob has the answer.", "Bob, watch out for that car!", "Is Bob still in the office?",
    "Thanks for everything, Bob.", "Bob, you won the first prize!", "Send my regards to Bob.",
    "Bob, do you want some coffee?", "Everything depends on Bob now.", "Bob, what's your opinion?",
    "I will meet Bob at the station.", "Bob, your flight is delayed.", "Can Bob handle this task?",
    "Bob, please turn off the light.", "Is Bob the manager here?", "I am looking for Bob's house.",
    "Bob, did you enjoy the movie?", "Wait for Bob at the entrance.", "Bob, I need your signature.",
    "Tell Bob that I called.", "Bob, don't forget your umbrella.", "Is Bob coming to the party?",
    "Bob, can you repeat that?", "I will check with Bob first.", "Bob, you look great today!",
    "Is Bob okay with this plan?", "Bob, let's go for a walk.", "Nobody knows Bob better than I do."
]

# --- 50 条不含 "Bob" 的语音 (False) ---
# 包含干扰词：Rob, Pop, Bub, Job, Barb 等
FALSE_TEMPLATES = [
    "Hello, is Alice there?", "Hey Rob, can you hear me?", "I'm calling to speak with Steve.",
    "Charlie, please pick up the phone.", "Is that you, Pop?", "Good morning, Dave.",
    "Tell Mary I'm waiting.", "Is Robert available?", "I am looking for Bill.",
    "Can I talk to Bub for a second?", "Job is very important to me.", "Where did Barb go?",
    "I saw Rob at the park yesterday.", "Please give this to John.", "Is the light on?",
    "We need to leave now.", "Is the manager available for a meeting?", "Ask her about the report.",
    "Did you see the email?", "That sounds like a good plan.", "Tell them the package arrived.",
    "Can you help me with this?", "I haven't seen her lately.", "Call them as soon as possible.",
    "Are you busy right now?", "It's time to join us.", "I believe he has the answer.",
    "Watch out for that car!", "Is he still in the office?", "Thanks for everything.",
    "You won the first prize!", "Send my regards to them.", "Do you want some coffee?",
    "Everything depends on this now.", "What's your opinion?", "I will meet you at the station.",
    "The flight is delayed.", "Can she handle this task?", "Please turn off the light.",
    "Is the boss here?", "I am looking for the house.", "Did you enjoy the movie?",
    "Wait at the entrance.", "I need your signature.", "Tell them that I called.",
    "Don't forget your umbrella.", "Is she coming to the party?", "Can you repeat that?",
    "I will check it first.", "You look great today!", "Are you okay with this plan?"
]

async def get_en_voices():
    """获取所有英语配音员"""
    voices = await edge_tts.VoicesManager.create()
    return [v["Name"] for v in voices.find(Language="en")]

def mix_audio(voice_wav_path, background_wav_path, output_path):
    """将语音随机混合进背景噪音"""
    voice = AudioSegment.from_file(voice_wav_path)
    background = AudioSegment.from_file(background_wav_path)

    # 统一采样率为 16000Hz (AST模型标准)
    voice = voice.set_frame_rate(16000).set_channels(1)
    background = background.set_frame_rate(16000).set_channels(1)

    # 确保背景刚好 10 秒
    background = background[:TARGET_DURATION_MS]

    # 计算随机插入位置 (0 到 10s - 语音时长)
    if len(voice) >= TARGET_DURATION_MS:
        # 如果语音太长（基本不可能），截断语音
        voice = voice[:TARGET_DURATION_MS-100]
    
    max_start = TARGET_DURATION_MS - len(voice)
    start_pos = random.randint(0, max_start)

    # 混合：语音音量保持原样，背景音如果太吵可以减弱 (-5 dB)
    combined = background.overlay(voice, position=start_pos)
    
    combined.export(output_path, format="wav")

async def run_generation():
    # 1. 初始化
    if not os.path.exists(NOISE_FILE):
        print(f"错误：找不到背景音文件 {NOISE_FILE}")
        return

    os.makedirs(TRUE_DIR, exist_ok=True)
    os.makedirs(FALSE_DIR, exist_ok=True)
    
    voices = await get_en_voices()
    print(f"准备就绪！可用人声: {len(voices)} 种。")

    # 确保循环次数不超过模板数量
    num_samples = min(len(TRUE_TEMPLATES), len(FALSE_TEMPLATES), TOTAL_SAMPLES_PER_CLASS)

    for i in range(num_samples):
        # 1. 每次随机选一种声音，但按顺序选模板
        v_true = random.choice(voices)
        v_false = random.choice(voices)
        
        t_true = TRUE_TEMPLATES[i]   # 按索引取值，保证不重复
        t_false = FALSE_TEMPLATES[i] # 按索引取值，保证不重复

        # --- 生成 True 样本 ---
        temp_t = f"temp_t_{i}.wav"
        comm_t = edge_tts.Communicate(t_true, v_true)
        await comm_t.save(temp_t)
        mix_audio(temp_t, NOISE_FILE, os.path.join(TRUE_DIR, f"bob_sample_{i}.wav"))
        
        # --- 生成 False 样本 ---
        temp_f = f"temp_f_{i}.wav"
        comm_f = edge_tts.Communicate(t_false, v_false)
        await comm_f.save(temp_f)
        mix_audio(temp_f, NOISE_FILE, os.path.join(FALSE_DIR, f"other_sample_{i}.wav"))

        if (i + 1) % 10 == 0:
            print(f"进度: {i + 1}/{num_samples} 组不重复数据已生成")
        
        # 清理临时文件
        if os.path.exists(temp_t): os.remove(temp_t)
        if os.path.exists(temp_f): os.remove(temp_f)

    print(f"\n生成成功！True/False 文件夹各含有 {num_samples} 个独特样本。")

await run_generation()

准备就绪！可用人声: 47 种。
进度: 10/50 组不重复数据已生成
进度: 20/50 组不重复数据已生成
进度: 30/50 组不重复数据已生成
进度: 40/50 组不重复数据已生成
进度: 50/50 组不重复数据已生成

生成成功！True/False 文件夹各含有 50 个独特样本。


In [7]:
import os
import random
import asyncio
import edge_tts
from pydub import AudioSegment
import nest_asyncio

# 允许在交互式环境中运行异步代码
nest_asyncio.apply()

# --- 配置 ---
OUTPUT_ROOT = "A"
TRUE_DIR = os.path.join(OUTPUT_ROOT, "true")
FALSE_DIR = os.path.join(OUTPUT_ROOT, "false")
TARGET_DURATION_MS = 10000  # 固定 10 秒
VOICES_PER_TEMPLATE = 4      # 每个文本生成 4 种不同成年人声

# --- 实验语料 (True Templates) ---
TRUE_TEMPLATES = [
    "Yo, Bob.", "Bob, what's your opinion?", "Bob, do you want some coffee?",
    "Bob, I need you.", "Bob, I'm behind you.", "What's up Bob?",
    "Bob, right here.", "Bob, can you take your headphones off for a second?",
    "Bob, look at this.", "Alice, Bob, Henry.", "Bob, follow me.",
    "Hello, Bob.", "Bob, stand up.", "Bob, wait for me.",
    "Can you hear me, Bob?", "Bob, it's your turn.", "Bob, give me that pencil.",
    "Hey, Bob!", "Good morning, Bob.", "Bob, how are you?",
    "Bob, how is it going?", "Bob", "Bob, are you busy right now?",
    "Bob, can you help me with this?", "Bob, turn off the music and listen to me."
]

# --- 对照组语料 (False Templates) ---
FALSE_TEMPLATES = [
    "Yo, Rob!", "Joe, what's your opinion?", "Mike, do you want some coffee?",
    "Anna, I need you.", "Tom, I'm behind you.", "What's up Kate?",
    "Ben, right here.", "Lisa, can you take your headphones off for a second?",
    "Paul, look at this.", "Alice, Lucy, Henry", "Sam, follow me.",
    "Hello, Steve.", "Jack, stand up.", "Rose, wait for me.",
    "Can you hear me, Emma?", "Leo, it's your turn.", "Mia, give me that pencil.",
    "Hey, Alice.", "Good morning, Charlie.", "Pop, how are you?",
    "Davey, how is it going?", "Mary", "Bill, are you busy right now?",
    "Sarah, can you help me with this?", "John, turn off the music and listen to me."
]

async def get_en_voices():
    """获取英文人声并定向排除用户指定的儿童音及重口音"""
    voices_manager = await edge_tts.VoicesManager.create()
    all_en_voices = voices_manager.find(Language="en")
    
    # --- 定向排除黑名单 ---
    # 包含你听测后确定的儿童音和咬字不清的口音
    exclude_voices_list = [
        "en-GB-MaisieNeural",
        "en-IN-NeerjaExpressiveNeural",
        "en-IN-NeerjaNeural",
        "en-IN-PrabhatNeural",
        "en-KE-AsiliaNeural",
        "en-KE-ChilembaNeural",
        "en-NG-EzinneNeural",
        "en-NG-AbeoNeural",
        "en-TZ-ImaniNeural",
        "en-TZ-ElimuNeural",
        "en-US-AnaNeural"
    ]
    
    adult_voices = []
    for v in all_en_voices:
        # edge-tts 的 ShortName 格式为 'en-US-GuyNeural'
        short_name = v["ShortName"]
        
        # 如果当前声音在黑名单中，则跳过
        if short_name in exclude_voices_list:
            continue
            
        adult_voices.append(v["Name"])
    
    print(f"筛选完成：已定向排除 {len(exclude_voices_list)} 个特定声库。")
    print(f"剩余可用声库共 {len(adult_voices)} 个。")
    return adult_voices

def process_audio_timing(voice_wav_path, output_path):
    """处理音频时长：10秒静音背景，随机嵌入语音或截断"""
    voice = AudioSegment.from_file(voice_wav_path)
    voice = voice.set_frame_rate(16000).set_channels(1)
    
    voice_duration = len(voice)
    silence = AudioSegment.silent(duration=TARGET_DURATION_MS, frame_rate=16000)

    if voice_duration >= TARGET_DURATION_MS:
        final_audio = voice[:TARGET_DURATION_MS]
    else:
        max_start = TARGET_DURATION_MS - voice_duration
        start_pos = random.randint(0, max_start)
        final_audio = silence.overlay(voice, position=start_pos)

    final_audio.export(output_path, format="wav")

async def run_generation():
    os.makedirs(TRUE_DIR, exist_ok=True)
    os.makedirs(FALSE_DIR, exist_ok=True)
    
    all_adult_voices = await get_en_voices()
    
    tasks_config = [("true", TRUE_TEMPLATES, TRUE_DIR), ("false", FALSE_TEMPLATES, FALSE_DIR)]

    for label, templates, target_dir in tasks_config:
        print(f"\n[开始生成] 类别: {label.upper()}")
        for t_idx, text in enumerate(templates):
            selected_voices = random.sample(all_adult_voices, min(VOICES_PER_TEMPLATE, len(all_adult_voices)))
            
            for v_idx, voice in enumerate(selected_voices):
                temp_wav = f"temp_{label}_{t_idx}_{v_idx}.wav"
                
                try:
                    communicate = edge_tts.Communicate(text, voice)
                    await communicate.save(temp_wav)
                    
                    file_name = f"{label}_text{t_idx:02d}_v{v_idx}.wav"
                    process_audio_timing(temp_wav, os.path.join(target_dir, file_name))
                except Exception as e:
                    print(f"文本 {t_idx} 处发生异常: {e}")
                finally:
                    if os.path.exists(temp_wav): 
                        os.remove(temp_wav)
            
            if (t_idx + 1) % 5 == 0:
                print(f" 进度: {t_idx + 1}/{len(templates)} 组样本已完成...")

    print(f"\n✅ 公交广播数据集生成完毕！")
    print(f"数据存放于: {os.path.abspath(OUTPUT_ROOT)}")

if __name__ == "__main__":
    # 注意：在普通 .py 脚本中应使用 asyncio.run(run_generation())
    # 在 Jupyter/IPython 中直接 await run_generation()
    await run_generation()

筛选完成：已定向排除 11 个特定声库。
剩余可用声库共 36 个。

[开始生成] 类别: TRUE
 进度: 5/25 组样本已完成...
 进度: 10/25 组样本已完成...
 进度: 15/25 组样本已完成...
 进度: 20/25 组样本已完成...
 进度: 25/25 组样本已完成...

[开始生成] 类别: FALSE
 进度: 5/25 组样本已完成...
 进度: 10/25 组样本已完成...
 进度: 15/25 组样本已完成...
 进度: 20/25 组样本已完成...
 进度: 25/25 组样本已完成...

✅ 公交广播数据集生成完毕！
数据存放于: /ihome/lshangguan/zhw227/original_dataset/A


In [1]:
import os
import asyncio
import edge_tts
import random
from pydub import AudioSegment
import nest_asyncio

nest_asyncio.apply()

# --- 配置 ---
TEST_OUTPUT_DIR = "D/voice_audit"
TEST_TEXT_TEMPLATE = "Hello, my name is {name}. I am a voice from {locale}. Our next stop is Times Square."
TARGET_DURATION_MS = 10000 

async def generate_audit_samples():
    os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
    
    # 获取所有英文声音
    voices_manager = await edge_tts.VoicesManager.create()
    all_en_voices = voices_manager.find(Language="en")
    
    print(f"找到共计 {len(all_en_voices)} 个英文声库。开始生成测试样本...")

    for idx, v in enumerate(all_en_voices):
        short_name = v["ShortName"]
        locale = v.get("Locale", "Unknown")
        gender = v.get("Gender", "Unknown")
        
        # 构造文件名：地区_性别_名称.wav (方便你在文件夹里按地区排序查看)
        file_name = f"{locale}_{gender}_{short_name}.wav"
        temp_wav = f"temp_audit_{idx}.wav"
        output_path = os.path.join(TEST_OUTPUT_DIR, file_name)
        
        # 构造测试文本
        text = TEST_TEXT_TEMPLATE.format(name=short_name, locale=locale)
        
        try:
            # 1. TTS 生成
            communicate = edge_tts.Communicate(text, short_name)
            await communicate.save(temp_wav)
            
            # 2. 标准化为 10 秒（沿用你之前的逻辑）
            voice = AudioSegment.from_file(temp_wav).set_frame_rate(16000).set_channels(1)
            silence = AudioSegment.silent(duration=TARGET_DURATION_MS, frame_rate=16000)
            
            if len(voice) >= TARGET_DURATION_MS:
                final_audio = voice[:TARGET_DURATION_MS]
            else:
                # 放在中间位置，方便听清楚
                final_audio = silence.overlay(voice, position=1000) 
            
            final_audio.export(output_path, format="wav")
            print(f"[{idx+1}/{len(all_en_voices)}] 已生成: {file_name}")
            
        except Exception as e:
            print(f"处理 {short_name} 时出错: {e}")
        finally:
            if os.path.exists(temp_wav):
                os.remove(temp_wav)

    print(f"\n✅ 所有样本已生成至: {os.path.abspath(TEST_OUTPUT_DIR)}")
    print("建议：你可以直接在文件夹里按名称排序，重点听 en-IN, en-PH, en-NG 等地区的样本。")

if __name__ == "__main__":
    await generate_audit_samples()

找到共计 47 个英文声库。开始生成测试样本...
[1/47] 已生成: en-AU_Male_en-AU-WilliamMultilingualNeural.wav
[2/47] 已生成: en-AU_Female_en-AU-NatashaNeural.wav
[3/47] 已生成: en-CA_Female_en-CA-ClaraNeural.wav
[4/47] 已生成: en-CA_Male_en-CA-LiamNeural.wav
[5/47] 已生成: en-HK_Female_en-HK-YanNeural.wav
[6/47] 已生成: en-HK_Male_en-HK-SamNeural.wav
[7/47] 已生成: en-IN_Female_en-IN-NeerjaExpressiveNeural.wav
[8/47] 已生成: en-IN_Female_en-IN-NeerjaNeural.wav
[9/47] 已生成: en-IN_Male_en-IN-PrabhatNeural.wav
[10/47] 已生成: en-IE_Male_en-IE-ConnorNeural.wav
[11/47] 已生成: en-IE_Female_en-IE-EmilyNeural.wav
[12/47] 已生成: en-KE_Female_en-KE-AsiliaNeural.wav
[13/47] 已生成: en-KE_Male_en-KE-ChilembaNeural.wav
[14/47] 已生成: en-NZ_Male_en-NZ-MitchellNeural.wav
[15/47] 已生成: en-NZ_Female_en-NZ-MollyNeural.wav
[16/47] 已生成: en-NG_Male_en-NG-AbeoNeural.wav
[17/47] 已生成: en-NG_Female_en-NG-EzinneNeural.wav
[18/47] 已生成: en-PH_Male_en-PH-JamesNeural.wav
[19/47] 已生成: en-PH_Female_en-PH-RosaNeural.wav
[20/47] 已生成: en-US_Female_en-US-AvaNeural.wav
[21/47] 已生成